# Debug Physics
Minimal notebook to sanity-check physics step functions (RBC / SFHelium).

In [ ]:
from phi.jax.flow import *
from sf_recon.physics.normal import boussinesq_step
from sf_recon.physics.helium import SFHelium_step
print('Imports ok')

In [ ]:
# Animated GT field check: vn/vs speed + streamlines + particles (animation)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors, patches # 引入 patches 用于绘制形状
from IPython.display import HTML, display

# 1. 加载数据
with np.load('../data/simulation/cyl_v0_recon.npz', allow_pickle=True) as data:
    v_gt_u = data.get('v_gt_u')
    v_gt_v = data.get('v_gt_v')
    vs_gt_u = data.get('vs_gt_u')
    vs_gt_v = data.get('vs_gt_v')
    gt_pts = data.get('gt')

def as_time_array(x):
    if x is None: return None
    try:
        a = np.array(x)
        if a.dtype == object:
            a = np.stack([np.asarray(e, dtype=float) for e in a])
        return a.astype(float)
    except Exception: return None

v_gt_u = as_time_array(v_gt_u)
v_gt_v = as_time_array(v_gt_v)
vs_gt_u = as_time_array(vs_gt_u)
vs_gt_v = as_time_array(vs_gt_v)
gt_pts = as_time_array(gt_pts)

if v_gt_u is None or v_gt_v is None:
    print('GT velocity fields not found in NPZ.')
else:
    # --- 领域参数设置 ---
    Lx, Ly = 0.02, 0.5
    Wy = 0.02
    Radius = 0.004 # 圆柱半径
    y_min, y_max = Ly/2 - Wy, Ly/2 + Wy

    sample = v_gt_u if v_gt_u is not None else vs_gt_u
    T, d1, d2 = sample.shape
    
    # 自动识别 H (对应 Ly) 和 W (对应 Lx)
    if d1 > d2:
        H, W = d1, d2
        is_transposed = False
    else:
        H, W = d2, d1
        is_transposed = True

    x = np.linspace(0, Lx, W)
    y = np.linspace(0, Ly, H)
    X, Y = np.meshgrid(x, y)

    # 找到 y 轴在局部窗口内的索引
    y_idx = np.where((y >= y_min) & (y <= y_max))[0]

    # --- 健壮的局部范围计算函数 ---
    def get_local_min_max(u_field, v_field, y_indices, transposed):
        if u_field is None or v_field is None: return 0.0, 1.0
        speed_full = np.sqrt(u_field**2 + v_field**2)
        if not transposed:
            local_data = speed_full[:, y_indices, :]
        else:
            local_data = speed_full[:, :, y_indices]
        
        _min, _max = np.nanmin(local_data), np.nanmax(local_data)
        return _min, _max

    vmin_vn, vmax_vn = get_local_min_max(v_gt_u, v_gt_v, y_idx, is_transposed)
    vmin_vs, vmax_vs = get_local_min_max(vs_gt_u, vs_gt_v, y_idx, is_transposed)

    # 打印调试信息
    print(f"Local Y Range: [{y_min:.4f}, {y_max:.4f}]")
    print(f"VN Local Speed Range: {vmin_vn:.6f} to {vmax_vn:.6f}")
    print(f"VS Local Speed Range: {vmin_vs:.6f} to {vmax_vs:.6f}")

    # --- 绘图设置 ---
    fig, axs = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    cmap = plt.cm.get_cmap('viridis')

    # 初始化背景 (imshow)
    init_vn = np.zeros((H, W))
    init_vs = np.zeros((H, W))
    
    # 使用 bilinear 插值使图像更平滑
    im0 = axs[0].imshow(init_vn, origin='lower', extent=[0, Lx, 0, Ly], 
                        cmap=cmap, vmin=vmin_vn, vmax=vmax_vn, interpolation='bilinear')
    im1 = axs[1].imshow(init_vs, origin='lower', extent=[0, Lx, 0, Ly], 
                        cmap=cmap, vmin=vmin_vs, vmax=vmax_vs, interpolation='bilinear')

    # --- 核心修改：添加圆柱体遮挡 ---
    # 定义圆心位置
    obs_center = (Lx / 2, Ly / 2)
    
    # 创建两个圆补丁 (Patch)
    # zorder=10 确保它覆盖在 imshow(z=0), streamplot(z=1), scatter(z=5) 之上
    obstacle0 = patches.Circle(obs_center, Radius, facecolor='gray', edgecolor='black', zorder=10)
    obstacle1 = patches.Circle(obs_center, Radius, facecolor='gray', edgecolor='black', zorder=10)
    
    # 将补丁添加到轴上
    axs[0].add_patch(obstacle0)
    axs[1].add_patch(obstacle1)
    # -----------------------------

    for ax in axs:
        ax.set_xlim(0, Lx)
        ax.set_ylim(y_min, y_max)
        ax.set_aspect('equal')
        ax.set_ylabel('y [m]')
        ax.set_xlabel('x [m]')

    cbar0 = fig.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)
    cbar0.set_label('vn speed (Local)')

    cbar1 = fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)
    cbar1.set_label('vs speed (Local)')

    s0, s1, scatter = [None], [None], [None]

    def _remove_streamplot(sp):
        if sp is None: return
        try: sp.lines.remove()
        except: pass
        try:
            for a in sp.arrows: a.remove()
        except: pass

    def update(i):
        # vn 场
        if v_gt_u is not None:
            u, v = v_gt_u[i], v_gt_v[i]
            if is_transposed: u, v = u.T, v.T
            speed = np.sqrt(u**2 + v**2)
            im0.set_data(speed)
            _remove_streamplot(s0[0])
            s0[0] = axs[0].streamplot(X, Y, u, v, color='k', density=5.5, linewidth=0.7)

        # vs 场
        if vs_gt_u is not None:
            u2, v2 = vs_gt_u[i], vs_gt_v[i]
            if is_transposed: u2, v2 = u2.T, v2.T
            speed2 = np.sqrt(u2**2 + v2**2)
            im1.set_data(speed2)
            _remove_streamplot(s1[0])
            s1[0] = axs[1].streamplot(X, Y, u2, v2, color='k', density=5.5, linewidth=0.7)

        # 粒子
        if gt_pts is not None:
            pts = np.asarray(gt_pts[i])
            mask = (pts[:, 1] >= y_min) & (pts[:, 1] <= y_max)
            vis_pts = pts[mask]
            if scatter[0] is not None: scatter[0].remove()
            # zorder=5，会被 zorder=10 的圆柱挡住
            scatter[0] = axs[0].scatter(vis_pts[:,0], vis_pts[:,1], 
                                        s=20, facecolors='white', edgecolors='black', zorder=5)

        axs[0].set_title(f'vn GT (Local Norm) t={i}')
        axs[1].set_title(f'vs GT (Local Norm) t={i}')
        return im0, im1

    num_frames = min(20, T)
    anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=250, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))